# Figuring out BCPNN

In [52]:
import sys
sys.path.insert(0,'..')

In [53]:
from vigipy import *
import pandas as pd

Read the data and only take what we need for the contingency table. Only get first few lines bc data is large

In [54]:
# Source - https://stackoverflow.com/a/69888274
# Posted by David Kaftan
# Retrieved 2026-05-25, License - CC BY-SA 4.0

from pyarrow.parquet import ParquetFile
import pyarrow as pa 

pf = ParquetFile(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat.parquet") 
first_n_rows = next(pf.iter_batches(batch_size = 1000)) 
ae_df = pa.Table.from_batches([first_n_rows]).to_pandas() 


In [55]:
# ae_df=pd.read_parquet(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat.parquet")
# ae_df.head()
#['AE', 'name', 'count'] ('date' is optional for longitudinal models)

drug_adverse = ae_df.groupby(["reaction_pt","drug_name"]).size().reset_index(name="count")

drug_adverse["AE"] = drug_adverse["reaction_pt"].astype(str)
drug_adverse["name"] = drug_adverse["drug_name"].astype(str)
drug_adverse["count"] = drug_adverse["count"].astype(int)

drug_adverse = drug_adverse[["AE", "name", "count"]]


Ora dovrebbe essere come lo vuole ui

In [56]:
drug_adverse.head()

,AE,name,count
0,ACUTE MYOCARDIAL INFARCTION,ATROPINE,1
1,ACUTE MYOCARDIAL INFARCTION,MORPHINE,1
2,ANHEDONIA,ACETYLSALICYLIC ACID SRT,1
3,ANHEDONIA,ALBUMIN (HUMAN),1
4,ANHEDONIA,ALTACE,1


Un sacco di problemi con la funzione che crea la contingency table usata in convert, per cui sovrascrivo con una che funzia x me ora:

In [57]:
import numpy as np
import vigipy.utils.data_prep as data_prep

def fixed_compute_contingency(data_frame, product_label, count_label, ae_label, margin_threshold):
    data_cont = pd.pivot_table(
        data_frame,
        values=count_label,
        index=product_label,
        columns=ae_label,
        aggfunc="sum",
        fill_value=0,
    )
    data_cont = data_cont.astype(float)
    data_cont.index = pd.Index(data_cont.index.astype(str).tolist())
    data_cont.columns = pd.Index(data_cont.columns.astype(str).tolist())

    # Usa boolean mask invece di np.where per evitare il problema PyArrow
    row_mask = np.sum(data_cont.values, axis=1) < margin_threshold
    col_mask = np.sum(data_cont.values, axis=0) < margin_threshold
    
    drop_rows = data_cont.index[row_mask]
    drop_cols = data_cont.columns[col_mask]
    data_cont = data_cont.drop(drop_rows)
    data_cont = data_cont.drop(drop_cols, axis=1)
    return data_cont

data_prep.compute_contingency = fixed_compute_contingency

In [58]:
data = convert(drug_adverse)

Funziona! caccio dentro bcpnn, the article on Iapatinib sets IC>0, IC025>0 (lower limit of 95% CI of CI greater than 0), N>=3

In [42]:
res=bcpnn(container=data, min_events=3, decision_metric="rank", ranking_statistic="quantile")
res.all_signals.head()
# volendo posso esportare su excel
# res.all_signals.to_excel(r"C:\Users\Admin\drug-safety-signal-detection\results\bcpnn_signals.xlsx", index=False)

,Product,Adverse Event,Count,Expected Count,quantile,count/expected,product margin,event margin,fdr,FNR,Se,Sp
254,TRASYLOL,INJURY,6.0,3.774,-0.788568,1.589825,102.0,37.0,0.529374,0.482411,0.354408,0.659901
474,TRASYLOL,RENAL FAILURE,9.0,8.364,-0.982986,1.076040,102.0,82.0,0.528593,0.499479,0.648254,0.363110
392,TRASYLOL,PAIN,9.0,8.466,-0.997974,1.063076,102.0,83.0,0.532132,0.482489,0.505519,0.509619
146,TRASYLOL,EMOTIONAL DISTRESS,9.0,8.568,-1.012809,1.050420,102.0,84.0,0.546723,0.478744,0.237456,0.770262
65,TRASYLOL,ANXIETY,9.0,8.568,-1.012809,1.050420,102.0,84.0,0.548408,0.479957,0.118287,0.889275


Filtro il lower threshold del quantile a >0 tanto è un dataframe pandas, che mi permette anche di creare i dataframeini per intensità del segnale (vedi tabella blu sulla bibbia)

In [59]:
res.all_signals[res.all_signals["quantile"] > 0]
# weak_signals=article_signals[article_signals["quantile"].isin(range(0,1.5))]

,Product,Adverse Event,Count,Expected Count,quantile,count/expected,product margin,event margin,fdr,FNR,Se,Sp


nn ce ne sono...... questo perchè l'IC che intendono i regaz di vigipy forse non è lo stesso che intendono quelli dell'articolo.  quando passo ranking_statistic='quantile' decido di rankare i segnali secondo il 2.5% quantile of the IC, che è lo stesso criterio che c'è nell'articolo.

Potrebbe anche essere che semplicemente non ci siano segnali significativi nelle prime mille righe. Provo a fare una run su tutto il parquet

In [60]:
tot_ae_df=pd.read_parquet(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat.parquet")
tot_drug_adverse = tot_ae_df.groupby(["reaction_pt","drug_name"]).size().reset_index(name="count")
tot_drug_adverse["AE"] = tot_drug_adverse["reaction_pt"].astype(str)
tot_drug_adverse["name"] = tot_drug_adverse["drug_name"].astype(str)
tot_drug_adverse["count"] = tot_drug_adverse["count"].astype(int)
tot_drug_adverse = tot_drug_adverse[["AE", "name", "count"]]

Converto (ci mette un po')

In [ ]:
tot_data = convert(tot_drug_adverse)

In [ ]:
tot_res=bcpnn(container=tot_data, min_events=3, decision_metric="rank", ranking_statistic="quantile")

In [ ]:
tot_res.all_signals[res.all_signals["quantile"] > 0]

,Product,Adverse Event,Count,Expected Count,quantile,count/expected,product margin,event margin,fdr,FNR,Se,Sp


## Volendo posso rendere il modello longitudinale:

In [33]:
LM = LongitudinalModel(ae_df, 'YE')
LM.run(bcpnn, include_gaps=False, decision_metric='rank', ranking_statistic='quantile')

#Change model time slice to quarterly
LM.regroup_dates('Q')
LM.run(bcpnn, include_gaps=False, decision_metric='rank', ranking_statistic='quantile')

#LM produces a list of timestamps and results
for timestamp, result in LM.results:
    print("Signals produced prior to {0}:".format(timestamp))
    print(result.signals.head())

KeyError: 'date'